# Multi-camera depth on iPhone 15 Pro Max

Compare depth from the iPhone's sensors and from recent papers, against the LiDAR depth camera:

| Method | Input | Source |
|---|---|---|
| `apple_stereo` | wide + ultra-wide (Apple's dual-camera disparity) | AVFoundation |
| `sgbm` | wide + ultra-wide, rectified here | classic semi-global matching (OpenCV) |
| `foundation_stereo` | wide + ultra-wide, rectified here | FoundationStereo, CVPR 2025 |
| `depth_pro` | wide only | Depth Pro (Apple), ICLR 2025 |
| `prompt_da` | wide + LiDAR as prompt | Prompt Depth Anything, CVPR 2025 |

Workflow: capture in **DepthKit › Settings › Developer › Lab Capture** (tripod, a label like `box-1.5m-night`, tape distance to the center), run `./pull_captures.sh`, then run this notebook. Install learned models with `./setup.sh <name>`.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from depthlab import io, stereo, metrics, viz, low_light
from depthlab.evaluate import evaluate

DATA = Path("../data")
captures = io.list_captures(DATA)
[c.name for c in captures]

## 1. One capture: images and sensor depth

In [ ]:
cap = io.load_capture(captures[0])
print(cap.label, "| tape:", cap.tape_distance, "m | lenses:", list(cap.images))
viz.show_images({k: v for k, v in cap.images.items()});
viz.show_depths({"LiDAR": cap.lidar.depth, "Apple stereo": cap.apple_stereo.depth});

## 2. Rectify wide + ultra-wide
Matching points should sit on the same green line. The cameras are ~1 cm apart, so disparities are small.

In [ ]:
pair = stereo.rectify(cap.images["wide"], cap.calibrations["wide"], cap.images["ultrawide"], cap.calibrations["ultrawide"])
print(f"baseline {pair.baseline*1000:.1f} mm, focal {pair.focal:.0f} px, rotated={pair.rotated}, reference_is_left={pair.reference_is_left}")
print(f"disparity at 1 m ≈ {pair.focal*pair.baseline:.1f} px, at 3 m ≈ {pair.focal*pair.baseline/3:.1f} px")
viz.show_rectified(pair.left, pair.right);

## 3. Evaluate all captures
Add `foundation_stereo`, `depth_pro`, `prompt_da` after installing them. `enhance=True` applies low-light enhancement to dark captures.

In [ ]:
METHODS = ["apple_stereo", "sgbm"]          # + "depth_pro", "prompt_da", "foundation_stereo"
rows = evaluate(DATA, METHODS, enhance=False, out=Path("../results"))
df = pd.DataFrame(rows)
df.pivot_table(index="method", values=["abs_rel", "rmse", "delta1", "center_vs_tape_m"], aggfunc="mean").round(3)

## 4. Day vs night

In [ ]:
df.pivot_table(index="method", columns="dark", values="abs_rel", aggfunc="mean").round(3)

## 5. Error vs distance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
name = captures[0].name
gt = np.load(f"../results/{name}__lidar.npy")
for m in METHODS:
    p = Path(f"../results/{name}__{m}.npy")
    if p.exists():
        viz.error_vs_distance(np.load(p), gt, ax=ax, label=m)
ax.legend();